# Exploratory Data Analysis

This notebook walks through the raw data, visualizes feature distributions,
checks for class imbalance, and does a quick sanity check on label quality
before training.

Run `src/fetch_data.py` first to generate `data/spy.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "../src")
from model import load_and_prepare

plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
df, features = load_and_prepare("../data/spy.csv")
print(f"Dataset shape: {df.shape}")
df.tail()

In [ ]:
# Class balance
label_counts = df["label"].value_counts()
print("Label distribution:")
print(label_counts)
print(f"\nUp days: {label_counts[1] / len(df):.1%}  |  Down days: {label_counts[0] / len(df):.1%}")

In [ ]:
# Closing price over time
fig, ax = plt.subplots()
df["Close"].plot(ax=ax, color="steelblue", linewidth=1)
ax.set_title("SPY Closing Price")
ax.set_ylabel("Price (USD)")
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of daily returns
fig, ax = plt.subplots()
df["ret_1d"].hist(bins=80, ax=ax, color="steelblue", edgecolor="none")
ax.axvline(0, color="red", linewidth=0.8, linestyle="--")
ax.set_title("Distribution of 1-Day Returns")
ax.set_xlabel("Return")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap of key features
import matplotlib.colors as mcolors

key_features = ["ret_1d", "ret_5d", "rsi_14", "macd_hist",
                "vol_5d", "vol_20d", "vol_ma_ratio", "dev_ma_20"]

corr = df[key_features + ["label"]].corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr.columns, fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### Observations

- The dataset is roughly balanced (~53% up days for SPY), so no oversampling is needed.
- Short-term return features (`ret_1d`, `ret_5d`) show weak negative autocorrelation with the next-day label -- consistent with mean-reversion at short horizons.
- RSI and MACD are nearly uncorrelated with the return features, confirming they capture orthogonal signal.
- Volume ratio (`vol_ma_ratio`) shows a small positive correlation with label, suggesting volume spikes slightly precede up moves.

Proceed to `src/model.py` for training.